# VII. Regularization and Shrinkage

- Compare at least two models (e.g., simple vs regularized)
- Explain how shrinkage affects performance

**Predictive Question:** Can we improve GPA prediction over a baseline linear regression model by applying regularization?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load data and set up features and target
data = pd.read_csv("../data/prepared_student_data.csv").dropna()

X = data.drop(columns=['semester_gpa', 'cumulative_gpa'])
y = data['cumulative_gpa']

# Train/test split — same random state as prediction notebook for consistency
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features — required for Ridge and Lasso so regularization penalizes fairly
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples: {X_train_sc.shape[0]}")
print(f"Test samples:     {X_test_sc.shape[0]}")
print(f"Features:         {X_train_sc.shape[1]}")

### Baseline Model: OLS Linear Regression

Ordinary Least Squares (OLS) is the unregularized baseline. It minimizes the sum of squared residuals with no penalty on coefficient size, which can lead to **overfitting** when there are many features relative to the signal in the data.

In [ ]:
ols = LinearRegression()
ols.fit(X_train_sc, y_train)
y_pred_ols = ols.predict(X_test_sc)

mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols  = r2_score(y_test, y_pred_ols)

print(f"OLS  —  MSE: {mse_ols:.4f}  |  R²: {r2_ols:.4f}")

### Ridge Regression (L2 Regularization)

Ridge adds an L2 penalty to the loss function — the sum of squared coefficients multiplied by a tuning parameter α:

$$\text{Loss} = \sum_{i}(y_i - \hat{y}_i)^2 + \alpha \sum_{j} \beta_j^2$$

This shrinks all coefficients toward zero but never sets them exactly to zero. We use cross-validation (`RidgeCV`) to select the best α from a grid of candidates.

In [ ]:
alphas = np.logspace(-3, 3, 100)

ridge = RidgeCV(alphas=alphas, cv=5)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge  = r2_score(y_test, y_pred_ridge)

print(f"Ridge —  MSE: {mse_ridge:.4f}  |  R²: {r2_ridge:.4f}  |  Best α: {ridge.alpha_:.4f}")

### Lasso Regression (L1 Regularization)

Lasso adds an L1 penalty — the sum of absolute coefficient values:

$$\text{Loss} = \sum_{i}(y_i - \hat{y}_i)^2 + \alpha \sum_{j} |\beta_j|$$

Unlike Ridge, Lasso can shrink coefficients **all the way to zero**, effectively performing automatic feature selection. Features with zero coefficients are excluded from the model entirely.

In [ ]:
lasso = LassoCV(alphas=alphas, cv=5, max_iter=10000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso  = r2_score(y_test, y_pred_lasso)

n_zero = np.sum(lasso.coef_ == 0)
print(f"Lasso —  MSE: {mse_lasso:.4f}  |  R²: {r2_lasso:.4f}  |  Best α: {lasso.alpha_:.4f}")
print(f"         Coefficients zeroed out: {n_zero} of {len(lasso.coef_)}")

### Comparing Model Performance

In [ ]:
results = pd.DataFrame({
    'Model': ['OLS (baseline)', 'Ridge (L2)', 'Lasso (L1)'],
    'MSE':   [mse_ols, mse_ridge, mse_lasso],
    'R²':    [r2_ols,  r2_ridge,  r2_lasso]
})
print(results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

models = ['OLS', 'Ridge', 'Lasso']
colors = ['steelblue', 'darkorange', 'seagreen']

# MSE bar chart
axes[0].bar(models, [mse_ols, mse_ridge, mse_lasso], color=colors, alpha=0.8)
axes[0].set_title('Test MSE by Model')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_ylim(0, max(mse_ols, mse_ridge, mse_lasso) * 1.2)

# R² bar chart
axes[1].bar(models, [r2_ols, r2_ridge, r2_lasso], color=colors, alpha=0.8)
axes[1].set_title('Test R² by Model')
axes[1].set_ylabel('R²')

plt.tight_layout()
plt.show()

### Effect of Shrinkage on Coefficients

The clearest way to see what regularization does is to plot the coefficients from all three models side by side. Ridge shrinks coefficients toward zero but keeps all of them; Lasso pushes some all the way to zero.

In [ ]:
feature_names = X.columns.tolist()
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'OLS':     ols.coef_,
    'Ridge':   ridge.coef_,
    'Lasso':   lasso.coef_
}).set_index('Feature')

# Sort by absolute OLS coefficient for readability
coef_df = coef_df.reindex(coef_df['OLS'].abs().sort_values(ascending=True).index)

fig, ax = plt.subplots(figsize=(8, 9))
y_pos = np.arange(len(coef_df))
width = 0.28

ax.barh(y_pos - width, coef_df['OLS'],   width, label='OLS',   color='steelblue', alpha=0.85)
ax.barh(y_pos,         coef_df['Ridge'], width, label='Ridge', color='darkorange', alpha=0.85)
ax.barh(y_pos + width, coef_df['Lasso'], width, label='Lasso', color='seagreen',  alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(coef_df.index, fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coefficients by Model (Standardized Features)')
ax.set_xlabel('Coefficient Value')
ax.legend()
plt.tight_layout()
plt.show()

#### Interpretation of Shrinkage

The coefficient plot shows the core effect of regularization. OLS assigns large coefficients wherever it can reduce training error, even if those large values reflect noise rather than a true signal. Ridge pulls all coefficients toward zero proportionally — large coefficients shrink more than small ones — reducing the model's sensitivity to any single feature. Lasso goes further, zeroing out the weakest predictors entirely and producing a sparser, more interpretable model.

When regularized models perform better than OLS on test data it means OLS was overfitting — learning patterns in the training set that did not generalize. When all three models perform similarly it suggests the data does not have a strong overfitting problem, and the regularization penalty is doing little harm or help.

### Bias–Variance Tradeoff

Regularization is a direct application of the **bias–variance tradeoff**. OLS is a low-bias, high-variance estimator: it fits the training data closely but can vary a lot across different samples. Adding a penalty (α) introduces bias — the model no longer fits the training data as tightly — but reduces variance, making predictions more stable on new data.

The optimal α found by cross-validation is the point where this tradeoff is best balanced: enough shrinkage to reduce overfitting, but not so much that the model becomes too simple to capture the real signal in the data.